In [ ]:
import os
import numpy as np
import pandas as pd
import motmetrics as mm
import json

# NOTE: motmetrics library only worked with NumPy versions >2.0 despite the requirements.txt for this project specifying numpy==2.2.6
# For this notebook, I used 1.26.4, you can just install the version and swap between them as needed using `pip install numpy==<version>`

# TODO: add evaluation for the second deep learning pipeline once Adit uploads to GitHub. Rerun if deep learning methods are improved.

In [ ]:
def load_mot_file(filepath):
    """
    Load MOT format file into a pandas DataFrame.
    MOT format: <frame>, <id>, <bb_left>, <bb_top>, <bb_width>, <bb_height>, <conf>, <x>, <y>, <z>
    """
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return pd.DataFrame(columns=['frame', 'id', 'x', 'y', 'width', 'height', 'conf'])
    
    # load MOT format file
    data = np.loadtxt(filepath, delimiter=',', dtype=np.float64)
    
    if len(data) == 0:
        return pd.DataFrame(columns=['frame', 'id', 'x', 'y', 'width', 'height', 'conf'])
    
    # handle single row case
    if data.ndim == 1:
        data = data.reshape(1, -1)
    
    # create dataframe with relevant columns
    df = pd.DataFrame({
        'frame': data[:, 0].astype(int),
        'id': data[:, 1].astype(int),
        'x': data[:, 2],
        'y': data[:, 3],
        'width': data[:, 4],
        'height': data[:, 5],
        'conf': data[:, 6] if data.shape[1] > 6 else np.ones(len(data))
    })
    
    return df

def convert_bbox_to_xyxy(bbox):
    """
    Convert bounding box from [x, y, width, height] to [x1, y1, x2, y2]
    """
    x, y, width, height = bbox
    return [x, y, x + width, y + height]

In [3]:
def evaluate_sequence(ground_truth_path, hypothesis_path, sequence_name):
    """
    Evaluate tracking results for a single sequence using motmetrics. Returns a dictionary of evaluation metrics.

    Parameters:
    ground_truth_path : str
        Path to the ground truth MOT file.
    hypothesis_path : str
        Path to the hypothesis MOT file.
    sequence_name : str
        Name of the sequence for reporting.
    """
    # load ground truth and hypothesis files
    ground_truth_df = load_mot_file(ground_truth_path)
    hypothesis_df = load_mot_file(hypothesis_path)
    
    if len(ground_truth_df) == 0:
        print(f"[{sequence_name}] empty ground truth file")
        return None
    
    if len(hypothesis_df) == 0:
        print(f"[{sequence_name}] empty hypothesis file")
        return None
    
    # create accumulator
    accumulator = mm.MOTAccumulator(auto_id=True)
    
    # get all unique frames from both ground truth and hypothesis
    all_frames = sorted(set(ground_truth_df['frame'].unique()) | set(hypothesis_df['frame'].unique()))
    
    # process each frame
    for frame_id in all_frames:
        # get ground truth detections for this frame
        gt_frame = ground_truth_df[ground_truth_df['frame'] == frame_id]
        gt_ids = gt_frame['id'].values
        gt_boxes = gt_frame[['x', 'y', 'width', 'height']].values
        
        # get hypothesis detections for this frame
        hyp_frame = hypothesis_df[hypothesis_df['frame'] == frame_id]
        hyp_ids = hyp_frame['id'].values
        hyp_boxes = hyp_frame[['x', 'y', 'width', 'height']].values
        
        # compute distance matrix (using IoU distance)
        if len(gt_boxes) > 0 and len(hyp_boxes) > 0:
            # convert boxes to [x1, y1, x2, y2] format for IoU calculation
            gt_boxes_xyxy = np.array([convert_bbox_to_xyxy(box) for box in gt_boxes])
            hyp_boxes_xyxy = np.array([convert_bbox_to_xyxy(box) for box in hyp_boxes])
            
            # compute IoU distance matrix (1 - IoU)
            distances = mm.distances.iou_matrix(gt_boxes_xyxy, hyp_boxes_xyxy, max_iou=0.5)
        else:
            distances = np.empty((len(gt_ids), len(hyp_ids)))
        
        # update accumulator
        accumulator.update(
            gt_ids,
            hyp_ids,
            distances
        )
    
    # compute metrics
    metrics_host = mm.metrics.create()
    summary = metrics_host.compute(
        accumulator,
        metrics=['num_frames', 'mota', 'motp', 'idf1', 'num_switches', 
                 'num_false_positives', 'num_misses', 'precision', 'recall'],
        name=sequence_name
    )
    
    # extract metrics into dictionary
    results = {
        'sequence_name': sequence_name,
        'num_frames': int(summary['num_frames'].values[0]),
        'mota': float(summary['mota'].values[0]),
        'motp': float(summary['motp'].values[0]),
        'idf1': float(summary['idf1'].values[0]),
        'num_switches': int(summary['num_switches'].values[0]),
        'num_false_positives': int(summary['num_false_positives'].values[0]),
        'num_misses': int(summary['num_misses'].values[0]),
        'precision': float(summary['precision'].values[0]),
        'recall': float(summary['recall'].values[0])
    }
    
    return results

In [4]:
def evaluate_all_sequences(ground_truth_dir, hypothesis_dir, output_file, method_name):
    """
    Evaluate all sequences in a directory and save results to a JSON file

    Parameters: 
    ground_truth_dir : str
        Directory containing ground truth sequences
    hypothesis_dir : str
        Directory containing hypothesis sequences
    output_file : str
        Path to save evaluation results JSON file
    method_name : str
        Name of the tracking method being evaluated
    """
    # find all sequences with ground truth
    sequences = []
    for item in os.listdir(ground_truth_dir):
        sequence_dir = os.path.join(ground_truth_dir, item)
        if os.path.isdir(sequence_dir):
            gt_file = os.path.join(sequence_dir, 'gt', 'gt.txt')
            if os.path.exists(gt_file):
                sequences.append((item, gt_file))
    
    print(f"Found {len(sequences)} sequences with ground truth")
    
    # evaluate each sequence
    all_results = []
    for sequence_name, gt_path in sequences:
        # construct hypothesis path - handle flat structure for yolov11
        hyp_path = os.path.join(hypothesis_dir, f"{sequence_name}.txt")
        
        # if not found, try nested structure
        if not os.path.exists(hyp_path):
            hyp_path = os.path.join(hypothesis_dir, sequence_name, f"{sequence_name}.txt")
        
        if not os.path.exists(hyp_path):
            print(f"[{sequence_name}] Warning: Hypothesis file not found")
            continue
        
        print(f"\nEvaluating sequence: {sequence_name}")
        results = evaluate_sequence(gt_path, hyp_path, sequence_name)
        
        if results:
            all_results.append(results)
            print(f"  MOTA: {results['mota']:.3f}")
            print(f"  MOTP: {results['motp']:.3f}")
            print(f"  IDF1: {results['idf1']:.3f}")
            print(f"  ID Switches: {results['num_switches']}")
    
    # compute average metrics
    if all_results:
        avg_metrics = {
            'method': method_name,
            'num_sequences': len(all_results),
            'avg_mota': np.mean([r['mota'] for r in all_results]),
            'avg_motp': np.mean([r['motp'] for r in all_results]),
            'avg_idf1': np.mean([r['idf1'] for r in all_results]),
            'total_switches': sum([r['num_switches'] for r in all_results]),
            'total_false_positives': sum([r['num_false_positives'] for r in all_results]),
            'total_false_negatives': sum([r['num_misses'] for r in all_results]),
            'avg_precision': np.mean([r['precision'] for r in all_results]),
            'avg_recall': np.mean([r['recall'] for r in all_results])
        }
        
        # save results
        output_data = {
            'summary': avg_metrics,
            'per_sequence': all_results
        }
        
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, 'w') as f:
            json.dump(output_data, f, indent=4)
        
        print(f"Number of sequences: {avg_metrics['num_sequences']}")
        print(f"Average MOTA: {avg_metrics['avg_mota']:.3f}")
        print(f"Average MOTP: {avg_metrics['avg_motp']:.3f}")
        print(f"Average IDF1: {avg_metrics['avg_idf1']:.3f}")
        print(f"Total ID Switches: {avg_metrics['total_switches']}")
        print(f"Total False Positives: {avg_metrics['total_false_positives']}")
        print(f"Total False Negatives: {avg_metrics['total_false_negatives']}")
        print(f"Average Precision: {avg_metrics['avg_precision']:.3f}")
        print(f"Average Recall: {avg_metrics['avg_recall']:.3f}")
        print(f"\nResults saved to: {output_file}")
        
        return output_data
    else:
        print("No sequences evaluated successfully")
        return None

In [ ]:
# Classical pipeline 
GROUND_TRUTH_DIR = "../data/soccer_side/test"
CLASSICAL_RESULTS_DIR = "../results/classical"
CLASSICAL_EVAL_OUTPUT = "../results/motmetrics_evaluation/classical_evaluation.json"

classical_results = evaluate_all_sequences(
    ground_truth_dir=GROUND_TRUTH_DIR,
    hypothesis_dir=CLASSICAL_RESULTS_DIR,
    output_file=CLASSICAL_EVAL_OUTPUT,
    method_name="Classical (SORT + Background Subtraction)"
)

Found 10 sequences with ground truth

Evaluating sequence: F_20220220_1_1890_1920
  MOTA: 0.548
  MOTP: 0.133
  IDF1: 0.537
  ID Switches: 104

Evaluating sequence: F_20220220_1_1920_1950
  MOTA: 0.125
  MOTP: 0.035
  IDF1: 0.238
  ID Switches: 39

Evaluating sequence: F_20220220_1_1680_1710
  MOTA: 0.287
  MOTP: 0.034
  IDF1: 0.395
  ID Switches: 48

Evaluating sequence: F_20220220_1_1770_1800
  MOTA: 0.298
  MOTP: 0.062
  IDF1: 0.412
  ID Switches: 66

Evaluating sequence: F_20220220_1_1950_1980
  MOTA: 0.092
  MOTP: 0.048
  IDF1: 0.229
  ID Switches: 28

Evaluating sequence: F_20220220_1_1830_1860
  MOTA: 0.579
  MOTP: 0.121
  IDF1: 0.457
  ID Switches: 95

Evaluating sequence: F_20220220_1_1740_1770
  MOTA: 0.156
  MOTP: 0.039
  IDF1: 0.290
  ID Switches: 39

Evaluating sequence: F_20220220_1_1860_1890
  MOTA: 0.304
  MOTP: 0.051
  IDF1: 0.434
  ID Switches: 47

Evaluating sequence: F_20220220_1_1800_1830
  MOTA: 0.451
  MOTP: 0.067
  IDF1: 0.514
  ID Switches: 69

Evaluating seque

In [ ]:
# YOLOv11 + BoT-SORT pipeline
YOLOV11_RESULTS_DIR = "../results/yolov11_botsort/mot_outputs"
YOLOV11_EVAL_OUTPUT = "../results/motmetrics_evaluation/yolov11_botsort_evaluation.json"

yolov11_results = evaluate_all_sequences(
    ground_truth_dir=GROUND_TRUTH_DIR,
    hypothesis_dir=YOLOV11_RESULTS_DIR,
    output_file=YOLOV11_EVAL_OUTPUT,
    method_name="YOLOv11 + BoT-SORT"
)


Evaluating YOLOv11 + BoT-SORT Method...
Found 10 sequences with ground truth

Evaluating sequence: F_20220220_1_1890_1920
  MOTA: 0.020
  MOTP: 0.068
  IDF1: 0.037
  ID Switches: 5

Evaluating sequence: F_20220220_1_1920_1950
  MOTA: -0.000
  MOTP: nan
  IDF1: 0.000
  ID Switches: 0

Evaluating sequence: F_20220220_1_1680_1710
  MOTA: 0.006
  MOTP: 0.044
  IDF1: 0.012
  ID Switches: 1

Evaluating sequence: F_20220220_1_1770_1800
  MOTA: 0.003
  MOTP: 0.413
  IDF1: 0.026
  ID Switches: 1

Evaluating sequence: F_20220220_1_1950_1980
  MOTA: -0.010
  MOTP: nan
  IDF1: 0.000
  ID Switches: 0

Evaluating sequence: F_20220220_1_1830_1860
  MOTA: -0.000
  MOTP: 0.137
  IDF1: 0.000
  ID Switches: 0

Evaluating sequence: F_20220220_1_1740_1770
  MOTA: -0.009
  MOTP: nan
  IDF1: 0.000
  ID Switches: 0

Evaluating sequence: F_20220220_1_1860_1890
  MOTA: -0.002
  MOTP: 0.055
  IDF1: 0.005
  ID Switches: 1

Evaluating sequence: F_20220220_1_1800_1830
  MOTA: -0.002
  MOTP: 0.193
  IDF1: 0.005
  I

/var/folders/2l/28jhqjz17l71xjd_wlwrg7hh0000gn/T/ipykernel_8139/3333823503.py:13: UserWarning: loadtxt: input contained no data: "../results/yolov11_botsort/mot_outputs/F_20220220_1_1710_1740.txt"
  data = np.loadtxt(filepath, delimiter=',', dtype=np.float64)
